# TP Lecture 5 : Modèles génératifs : VAE, GAN, flows, diffusion

Module 5. Modèles de petite taille : VAE, GAN et flow en quelques minutes sur CPU ; le DDPM (petit U-Net)
demande environ 15 minutes sur CPU (moins d'une minute sur GPU).
MNIST est chargé via scikit-learn (pas besoin de `torchvision`).

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml, make_moons

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X01 = torch.tensor(mnist.data[:60000] / 255.0, dtype=torch.float32)       # images dans [0,1]
y_train = torch.tensor(mnist.target[:60000].astype(int))

def batches(X, bs, shuffle=True):
    idx = torch.randperm(len(X)) if shuffle else torch.arange(len(X))
    for i in range(0, len(X), bs):
        yield X[idx[i:i + bs]]

def show_grid(imgs, title, n=16):
    imgs = imgs[:n].detach().cpu().clamp(0, 1).view(-1, 28, 28)
    fig, axes = plt.subplots(1, n, figsize=(n, 1.3))
    for a, im in zip(axes, imgs):
        a.imshow(im, cmap="gray"); a.axis("off")
    plt.suptitle(title); plt.show()

## Exercice 1 : VAE
1. Implémentez un VAE (encodeur MLP → $\mu,\log\sigma^2$ de dimension 2 ; décodeur MLP → logits de 784 pixels).
2. Perte : $-\mathrm{ELBO}$ = entropie croisée binaire (`binary_cross_entropy_with_logits`, somme) + $\mathrm{KL}(q_\phi(z|x)\|\mathcal N(0,I))$.
3. Entraînez 10 époques ; visualisez l'espace latent (couleur = chiffre) et le décodage d'une grille du latent.
4. Pourquoi la perte n'est-elle qu'une **borne** de la log-vraisemblance ?

In [ ]:
# (À compléter)

## Exercice 2 : GAN
1. Générateur MLP terminé par `Tanh` : mettez les images dans $[-1,1]$ (`X01 * 2 - 1`). Pourquoi est-ce indispensable ?
2. Discriminateur renvoyant des logits, `BCEWithLogitsLoss`, perte **non saturante** pour $G$.
3. Entraînez 20 époques en affichant des échantillons toutes les 5 époques. Les pertes mesurent-elles la qualité ?

In [ ]:
# (À compléter)

## Exercice 3 : Normalizing flow (RealNVP) sur des données 2D
Une transformation affine **fixe** $y=ax+b$ n'est pas un modèle génératif appris. Implémentez un vrai flow :
1. Couche de couplage affine : $y_1=x_1$, $y_2=x_2\odot e^{s(x_1)}+t(x_1)$ ; écrivez l'inverse et $\log|\det J|$.
2. Empilez 8 couches en alternant les coordonnées ; maximisez $\log p_X(x)=\log p_Z(f^{-1}(x))+\log|\det J_{f^{-1}}(x)|$
   sur `make_moons` (données standardisées).
3. Vérifiez numériquement l'inversibilité ; tracez la densité apprise et des échantillons.

In [ ]:
# (À compléter)

## Exercice 4 : Modèle de diffusion (DDPM) sur MNIST
1. Calculez $\bar\alpha_T$ pour $\beta_t$ linéaire de $10^{-4}$ à $0{,}02$, avec $T=100$ puis $T=1000$. Lequel choisir, et pourquoi ?
2. Écrivez `q_sample(x0, t, noise)` ; attention aux formes : `t` est de forme `(B,)`, `x0` de forme `(B, 784)`.
3. Réseau $\epsilon_\theta(x_t,t)$ : un petit U-Net convolutif qui reçoit **aussi** $t$ (encodage sinusoïdal ajouté
   aux canaux). Données dans $[-1,1]$. Essayez d'abord un MLP : que constatez-vous sur la perte et les échantillons ?
4. Entraînez avec la perte $\|\epsilon-\epsilon_\theta(x_t,t)\|^2$ (environ 6 époques).
5. Implémentez l'échantillonnage inverse
   $x_{t-1}=\frac{1}{\sqrt{\alpha_t}}\big(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\epsilon_\theta(x_t,t)\big)+\sqrt{\beta_t}\,z$ et affichez des échantillons.
6. Implémentez DDIM (50 pas, déterministe) et comparez au DDPM (1000 pas).
7. Visualisez le débruiteur de Tweedie $\hat x_0=(x_t-\sqrt{1-\bar\alpha_t}\,\epsilon_\theta)/\sqrt{\bar\alpha_t}$ pour différents $t$.

In [ ]:
# (À compléter)

## Comparaison
Comparez qualité, diversité, coût d'échantillonnage et accès à la vraisemblance des quatre modèles.